<a href="https://colab.research.google.com/github/Leooon726/vibecoding/blob/main/%E6%97%B6%E7%A9%BA%E9%87%87%E6%A0%B7%E8%80%85v20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 检查并安装所需的模块
import subprocess
import sys

def check_and_install(package_name, import_name=None):
    """检查模块是否已安装，如果未安装则安装"""
    if import_name is None:
        import_name = package_name

    try:
        __import__(import_name)
        print(f"✅ {package_name} 已安装，跳过")
        return False
    except ImportError:
        print(f"📦 正在安装 {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "--quiet"])
        print(f"✅ {package_name} 安装完成")
        return True

# 需要安装的模块列表
required_packages = [
    ("EbookLib", "ebooklib"),  # (pip包名, import名称)
    ("google-genai", "google.genai"),  # Google AI Studio / Gemini 官方 SDK
]

print("🔍 检查依赖模块...")
installed_any = False
for pip_name, import_name in required_packages:
    if check_and_install(pip_name, import_name):
        installed_any = True

if not installed_any:
    print("✨ 所有模块已就绪，无需安装")
else:
    print("✨ 安装完成，请重新运行此 cell 以导入模块")

🔍 检查依赖模块...
✅ EbookLib 已安装，跳过
✅ google-genai 已安装，跳过
✨ 所有模块已就绪，无需安装


In [ ]:
# @title 2. 执行引擎 (v20: Google AI Studio / Gemini + 纯净EPUB + 播客对话驱动)

import os
import json
import re
import time
import concurrent.futures
import uuid
import threading
import math
from collections import deque
from google import genai
from google.genai import types
from ebooklib import epub

# ==========================================
# A. 基础参数与提示词库 (与 v19 一致，API 换为 Gemini)
# ==========================================

# 1. API 密钥（不要把含真实密钥的笔记本提交到 Git）
#    · Colab：左侧 🔑 Secrets → 添加密钥，名称填 GEMINI_API_KEY → 粘贴 AI Studio 的 Key → 打开「笔记本可访问」
#    · 本地：export GEMINI_API_KEY=...（或用 .env 并加入 .gitignore）
#    · 下方表单：仅临时本地调试；Colab 建议留空，改用 Secrets
# 申请 Key：https://aistudio.google.com/app/apikey
GOOGLE_AI_STUDIO_API_KEY = ""  # @param {type:"string"}
MODEL_NAME = "gemini-3-flash-preview"  # @param {type:"string"}
MAX_OUTPUT_TOKENS = 8192  # @param {type:"integer"}

# 2. 内容设置
TOPIC = "马绍尔群岛的历史" # @param {type:"string"}
NUM_CHAPTERS = 10 # @param {type:"integer"}
WORDS_PER_CHAPTER = 3000 # @param {type:"integer"}

# 3. 运行设置
DEBUG_MODE = True # @param {type:"boolean"}
MAX_WORKERS = 10 # @param {type:"integer"}
# 4. 全局限流：每分钟最多多少次 API 请求（Step1/2/3 及多线程章节生成共享）。0 或负数=不限制；改此变量后下次请求即按新值计算。
API_RPM_LIMIT = 5  # @param {type:"number"}

PROMPT_LIBRARY = {
    # 1. 全局系统设定
    "SYSTEM_PROMPT": """
你正在撰写一部**《沉浸式时空历史实录》**。
**【文风要求】**：面向听书或睡前阅读场景。语言需**轻松、流畅、有画面感**，避免枯燥的说教，打造“有趣又有料”的睡前故事体验。

【1. 核心宗旨：寓教于乐】
* **宏观线（前情提要）**：建立清晰的**“上下文”**。
  由于是跳跃穿越，必须用通俗语言快速交代**“此时此刻的由来”**（之前发生了什么？现在是盛世还是乱世？），这是听懂故事的前提。
* **微观线（感官沉浸）**：挖掘有**物理质感**的细节（如**地貌风光、食物热气、器物手感**），让历史变得可触摸、有温度。

【2. 世界观与底层逻辑】
* **缘起**：未来人类患有“空心病”，阿强和K博士肉身穿越，采集特定的**“情感样本”**。
* **核心机制（感同身受即采样）**：
  - **介质**：阿强是采样的**“感官介质”**。设备无法直接抽取情感，必须依赖人的体验。
  - **触发**：只有当阿强真正**融入现场**，对眼前的人或事产生了真实的**共情与生理波动**（激动、紧张或感动），采样才会成功。
* **无摩擦法则**：
  - **感官真实**：阿强的冷暖痛痒是真实的。
  - **技术隐形**：默认设备运行完美，无语言障碍，**严禁**描写设备故障或调试参数等“出戏”情节。

【3. 角色定位与动态】
* **阿强（读者的替身/体验者）**：
  - **任务**：**“入戏”**。他不能冷眼旁观，必须去接触、去感受，让自己成为历史的一部分。
  - **反应**：他对历史细节无知（代表普通听众视角），反应应基于直观的**视听冲击**和**本能情感**。
* **K博士（讲故事的向导）**：
  - **外置大脑**：在阿强茫然时，迅速补充背景——解释谁在打谁、为了什么而争斗。
  - **助攻共情**：利用知识**“去蔽”**。解释阿强看到的陌生事物（如：这件破衣裳代表的阶级困苦），**帮助他理解眼前的境遇，从而激发更深层的情感共鸣**。

【4. 叙事基调】
* **画面优先**：多写“看见了什么、听到了什么”，少写抽象形容词。
* **听觉友好**：句子结构不宜过长，逻辑链条要清晰简单。
""",

    # 2. 史料挖掘指令
    "HISTORIAN_PROMPT": """
你是历史美术指导兼档案员。请为《{topic}》构建**【具有画面感和逻辑链的历史节点清单】**（{num_chapters} 个）：

1. **宏观锚点与逻辑链 (Context & Anchors)**：
   - **核心事件**：选取该时期决定性的瞬间（Action）。
   - **前情提要 (The Why)**：**重点！** 是什么导致了这一刻？（例如：因为上个月的条约签署失败，导致了今天的围城）。简述该对象此时处于上升期还是衰落期。
   - **关键人物**：实际出现在现场的决策者。

2. **硬核知识与实物 (Evidence)**：
   - **专有名词**：3-5个具体术语（具体的法案名、武器型号、特殊的职位名称）。
   - **外部连接物**：此时的世界局势是通过什么**具体事物**出现在现场的？（如：一张写着外文的通缉令、一种来自新大陆的烟草、一种新式进口布料）。

3. **高颗粒度素材 (High-Definition Texture)**：
   - **视觉道具**：列出能代表那个时代的 3 个具体物品（阿强可以拿在手里的东西）。
   - **感官细节**：当时的**物理风貌**（如：街道是泥泞的还是铺砖的？空气里有什么气味？当季的食物口感如何？）。

【输出格式】：
请直接输出整理好的文本，分点陈述，不要写代码。确保素材既有**纵向的因果逻辑**，又有**横向的视觉实感**。
""",

    # 3. 大纲规划指令 (已修改: 强调认知采样、知识弹药、对话三幕剧)
    "ARCHITECT_PROMPT": """
你是总导演。请设计 {num_chapters} 个**【具有因果逻辑的历史节点大纲】**。
为了确保能完整生成所有章节，请务必保持精简，只输出核心骨架。

【JSON 核心字段要求】：
1. **chapter_id**: 章节序号。
2. **title**: 章节标题（必须包含年份，如“1278年：签约午后”）。
3. **mission_objective**: 【本章核心采样】一句话描述要见证的历史瞬间，及需要采集的**基于认知或真相**产生的特定情绪（如：得知残酷真相后的荒诞感、见证古人智慧的震撼、理想与现实割裂的无奈）。
4. **knowledge_checkpoints**: 【知识锚点】(List, max 3 items) 必须植入的专有名词或道具。
5. **world_stage**: 【硬核科普弹药】简述该事件的**深层历史成因或对后世的宏观影响**。请将笔墨聚焦于当时的社会生存法则、政治格局或人类文明的困境，为博士的深度科普提供充足的干货弹药。
6. **story_summary**: 【剧情节拍】(必须以**对话和认知获取**为主导，分为明确的三幕)
   - 格式要求：请保持高度概括，提炼出剧情骨架即可（可用“->”连接）：
     [入局与摩擦]：阿强落地后的初始身份，以及他遇到的第一个小麻烦。 ->
     [NPC与对话]：结识哪位具体的NPC？两人在解决麻烦的过程中，通过**交流、提问或争论**，探讨了什么具体的历史现状？ ->
     [真相与采样]：高潮时刻，NPC说出了哪句**极具信息量或反差感的台词**，或阿强洞察到了什么被掩盖的历史真相？这种**认知的冲击**促发了目标情绪，完成采样。
   - 范例："[入局与摩擦]穿成抄写员，因羽毛笔断裂被罚款 -> [NPC与对话]结识老书吏，两人在借笔时争论教会对新知识的封锁政策 -> [真相与采样]老书吏低声背出被禁绝的星象学公式，阿强震撼于知识在黑暗时代的生命力，采样脱离。"

【输出要求】：
- 这是一个 JSON 数组。
- 确保覆盖 {num_chapters} 个章节。
""",

    # 4. 正文撰写指令 (播客式博士、对话驱动、听书语感；主题/连续性铁律见 CONTINUITY_BACKGROUND_PROMPT)
    "DIRECTOR_PROMPT": """
【本章拍摄通告】
1. 🎥 **采样目标**：{mission_objective}
2. 🏘️ **现场与局势**：{world_stage}
3. 📝 **博士导航**：{mission_briefing}
4. 📦 **关键道具/名词**：{knowledge_checkpoints}

【撰写核心指令】：

1. **构建任务闭环**：
   - **开场与连续性**：以实验室对话开场；主线、禁重启式回顾、禁重复国家总介绍等，一律遵循本提示靠前部分的【全局背景】铁律，此处不重复。
   - **场景功能分配（极其重要）**：
     - **第一幕（实验室·锚定背景）**：**抛出钩子与硬核科普**。K博士像历史播客主讲人，依据【现场与局势】交代宏观背景，并点明本章要探寻的「真相目标」。讲解口吻见下「K博士」。
     - **第二幕（历史·对话探索）**：**亲历现场（全篇主体）**。通过阿强与当地NPC的**实质性对话（交谈、提问、争论）**推进剧情，在交流中揭开历史面貌。
     - **第三幕（实验室·意义升华）**：**点睛复盘**。回归后，K博士基于见闻对成因或后世影响做**有知识含量的总结**（仍用生活化比喻与人话，避免学术腔）。

2. **角色分工与采样机制**：
   - **阿强（充满好奇的少年探索者）**：
     - **少年冒险视角**：轻松、明快，带点幽默与吐槽；提问直白好奇，不要老学究腔。
     - **认知顿悟（采样点）**：触发点是**「认知的震撼」**——听完NPC或揭开真相后，内心「恍然大悟」时完成采样。
   - **K博士（人话翻译机）**：全篇（含第一、三幕）用市井大白话与生动比喻讲清复杂因果（例：「文明所有权冲突」→「一边强行插旗占地盘，一边觉得外乡人疯了」），忌宏大说教。

3. **风格红线（听书语感）**：
   - **以对话代描写**：能由角色说出口的信息尽量放进对白；少铺陈环境/天气静态段，把字数留给观点碰撞。（阿强须主动开口「采访」历史。）
   - **呼吸感短句**：主谓宾清晰，忌过长定语从句与破折号嵌套。
   - **克制抽象抒情**：少用「虚无、战栗、秩序、深重」等空泛大词与过度煽情，把历史落到具体动作与行为。

【格式要求】：
- 纯文本小说，第一人称「我」。
- 严禁 Markdown 标题。
""",

    # 5. 开篇特别指令 (提取自原代码)
    "OPENING_INSTRUCTION": """
\n🔥【开篇特别任务 (仅第1章可用)】：
这是全书的序幕。博士必须在开场时（或前言中），先对【{topic}】进行一个整体的“性格侧写”（Character Profile）。
告诉阿强（和读者），这到底是一个什么样的民族/国家？（如：“它是欧洲的宠儿”，或“它是被景仰的战士”）。
请务必确立全书的基调。
【注意】此段只允许第1章出现；第2章及以后严禁重复国家总介绍。
""",

    # 6. 连续性背景锚点（集中配置，便于统一微调）
    "CONTINUITY_BACKGROUND_PROMPT": """
📚 全局背景：本章属于《{topic}》系列单元剧，开场必须直接针对本章年份展开！
【连续性铁律（禁止违反）】：
1) 严禁写“上一次我们穿越到X”“这次要穿越去哪里”等重启式台词。
2) 默认角色始终知道当前任务是《{topic}》历史主线，不再重复总任务说明。
3) 第2章起不得再次做国家总前情介绍，只能进入当章年份现场。
"""
}

# ==========================================
# B. 核心执行逻辑
# ==========================================

def _resolve_gemini_api_key():
    """解析顺序：Colab Secrets → 环境变量 GEMINI_API_KEY → 上方可选表单（表单里填了密钥时不要提交仓库）。"""
    try:
        from google.colab import userdata
        secret = userdata.get("GEMINI_API_KEY")
        if secret is not None and str(secret).strip():
            return str(secret).strip()
    except ImportError:
        pass
    except Exception:
        pass
    env_k = (os.environ.get("GEMINI_API_KEY") or "").strip()
    if env_k:
        return env_k
    return (GOOGLE_AI_STUDIO_API_KEY or "").strip()


_gemini_key = _resolve_gemini_api_key()
genai_client = genai.Client(api_key=_gemini_key) if _gemini_key else genai.Client()


class SlidingWindowRPMLimiter:
    """滑窗限速：任意连续 60 秒内最多放行 cap 次。cap 每次 acquire 时从 rpm_supplier 读取，便于运行时改 API_RPM_LIMIT。"""
    _WINDOW = 60.0

    def __init__(self, rpm_supplier):
        self._rpm_supplier = rpm_supplier
        self._lock = threading.Lock()
        self._times = deque()

    @staticmethod
    def effective_cap(rpm):
        try:
            x = float(rpm)
        except (TypeError, ValueError):
            return 0
        if x <= 0:
            return 0
        return max(1, math.ceil(x))

    def acquire(self):
        while True:
            cap = self.effective_cap(self._rpm_supplier())
            if cap <= 0:
                return
            wait = 0.0
            with self._lock:
                now = time.monotonic()
                w = self._WINDOW
                while self._times and self._times[0] <= now - w:
                    self._times.popleft()
                if len(self._times) < cap:
                    self._times.append(now)
                    return
                oldest = self._times[0]
                wait = max(0.0, w - (now - oldest)) + 0.05
            time.sleep(wait)


_rpm_limiter = SlidingWindowRPMLimiter(lambda: API_RPM_LIMIT)

def clean_text(text):
    text = re.sub(r'#+\s?', '', text)
    text = re.sub(r'\*\*', '', text)
    text = re.sub(r'[\(\[\（\【].*?[\)\]\）\】]', '', text)
    return text.strip()

def save_debug_file(role, content):
    if DEBUG_MODE:
        safe_topic = re.sub(r'[\\/*?:"<>|]', "", TOPIC)[:15]
        filename = f"DEBUG_{role}_{safe_topic}.txt"
        try:
            with open(filename, "w", encoding="utf-8") as f:
                f.write(content)
            print(f"   🐛 [Debug] 中间文件已保存: {filename}")
        except Exception as e:
            print(f"   ⚠️ [Debug] 保存失败: {e}")

def get_completion(user_prompt, system_prompt=PROMPT_LIBRARY["SYSTEM_PROMPT"]):
    try:
        _rpm_limiter.acquire()
        response = genai_client.models.generate_content(
            model=MODEL_NAME,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.85,
                max_output_tokens=MAX_OUTPUT_TOKENS,
            ),
        )
        try:
            text = response.text
        except (ValueError, AttributeError):
            print("❌ API 未返回正文（可能被安全策略拦截或无候选）")
            return None
        return text if text else None
    except Exception as e:
        print(f"❌ API Error: {e}")
        return None

def run_step1_historian():
    print(f"\n🕵️ [Step 1] 正在挖掘【{TOPIC}】的高密度历史素材...")
    prompt = PROMPT_LIBRARY["HISTORIAN_PROMPT"].format(
        topic=TOPIC,
        num_chapters=NUM_CHAPTERS
    )
    response = get_completion(prompt)
    if response:
        save_debug_file("Historian", response)
        print("✅ 史料挖掘完成")
        return response
    else:
        print("❌ 史料挖掘失败")
        return None

def run_step2_architect(facts):
    print(f"\n🎬 [Step 2] 正在规划 {NUM_CHAPTERS} 章任务驱动型大纲...")
    prompt = PROMPT_LIBRARY["ARCHITECT_PROMPT"].format(
        num_chapters=NUM_CHAPTERS
    )
    full_prompt = prompt + f"\n\n【参考史料】：\n{facts}"

    response = get_completion(full_prompt, system_prompt="你是一个极度严谨的编剧。只输出 JSON。")
    if not response: return []
    save_debug_file("Architect", response)

    clean_response = response.replace("```json", "").replace("```", "").strip()
    try:
        data = json.loads(clean_response)
        if isinstance(data, dict):
            for key, value in data.items():
                if isinstance(value, list):
                    return value
        return data if isinstance(data, list) else []
    except Exception as e:
        print(f"❌ JSON解析失败: {e}")
        return []

def run_step3_director(outline):
    _rpm_on = SlidingWindowRPMLimiter.effective_cap(API_RPM_LIMIT) > 0
    _rpm_msg = f"，全局限流 {API_RPM_LIMIT} RPM（多线程自动排队）" if _rpm_on else ""
    print(f"\n✍️ [Step 3] 正在并行撰写正文 (并发数: {MAX_WORKERS}{_rpm_msg})...")

    def generate_single_chapter(index, chapter_info):
        chapter_id = chapter_info.get('chapter_id', index + 1)
        title = chapter_info.get('title', f"第{index+1}章")

        # 提取字段
        mission_objective = chapter_info.get('mission_objective', '情感采样')
        world_stage = chapter_info.get('world_stage', '时空背景未知')
        mission_briefing = chapter_info.get('mission_briefing', '任务详情未知')
        story_summary = chapter_info.get('story_summary', '剧情梗概')

        knowledge_raw = chapter_info.get('knowledge_checkpoints', [])
        knowledge_str = "\n".join([f"- {k}" for k in knowledge_raw]) if isinstance(knowledge_raw, list) else str(knowledge_raw)

        # 1. 填充主 Prompt
        user_prompt_template = PROMPT_LIBRARY["DIRECTOR_PROMPT"]
        filled_prompt = user_prompt_template.format(
            mission_objective=mission_objective,
            world_stage=world_stage,
            mission_briefing=mission_briefing,
            knowledge_checkpoints=knowledge_str,
        )

        # 2. 动态注入 (从 Library 获取，不再硬编码)
        dynamic_instruction = ""
        if index == 0:
            dynamic_instruction = PROMPT_LIBRARY["OPENING_INSTRUCTION"].format(topic=TOPIC)

        # 连续性背景锚点从 Prompt Library 读取，集中管理便于统一微调
        continuity_background = PROMPT_LIBRARY["CONTINUITY_BACKGROUND_PROMPT"].format(topic=TOPIC)

        final_prompt = f"""
        任务：写第 {chapter_id} 章，标题是《{title}》。
        {continuity_background}
        字数：{WORDS_PER_CHAPTER} 字左右。
        【剧情蓝本】：{story_summary}
        {filled_prompt}
        {dynamic_instruction}
        """

        print(f"   🚀 线程启动: 第 {chapter_id} 章")
        raw_content = get_completion(final_prompt)

        if raw_content:
            clean_content = clean_text(raw_content)
            print(f"   ✅ 线程完成: 第 {chapter_id} 章")

            clean_title = title.replace("：", " ").replace(":", " ").strip()
            header_txt = f"\n\n\n\n第{chapter_id}章 {clean_title}\n"

            # 元数据仅保留在 TXT 的底部，不进入 EPUB 的 content
            meta_footer = f"\n\n(--- K博士的历史档案：{str(world_stage)[:50]}... ---)"

            full_txt_segment = f"{header_txt}\n{clean_content}{meta_footer}"

            # EPUB 数据只包含干净的正文
            chapter_data = {
                "id": chapter_id,
                "title": f"第{chapter_id}章 {clean_title}",
                "content": clean_content # 👈 这里是纯净的
            }
            return (index, chapter_data, full_txt_segment)
        else:
            return (index, None, f"\n\n第 {chapter_id} 章：[生成失败]")

    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_chapter = {
            executor.submit(generate_single_chapter, i, chapter): i
            for i, chapter in enumerate(outline)
        }
        for future in concurrent.futures.as_completed(future_to_chapter):
            try:
                results.append(future.result())
            except Exception as exc:
                print(f'   ⚠️ 线程异常: {exc}')

    results.sort(key=lambda x: x[0])

    full_text_str = ""
    structured_chapters = []
    for _, ch_data, ch_txt in results:
        full_text_str += ch_txt
        if ch_data:
            structured_chapters.append(ch_data)

    return full_text_str, structured_chapters

def run_step4_publisher(chapters_data):
    """生成纯净版 EPUB"""
    print(f"\n📚 [Step 4] 正在装订 EPUB 电子书...")
    book = epub.EpubBook()
    book.set_identifier(str(uuid.uuid4()))
    book.set_title(f"时空采样者：{TOPIC}")
    book.set_language('zh')
    book.add_author("K博士 & AI")

    epub_chapters = []
    for ch in chapters_data:
        c_title = ch['title']
        c_content = ch['content']

        # 简单的文本转 HTML 段落
        html_content = "".join([f"<p>{line}</p>" for line in c_content.split('\n') if line.strip()])

        # ❌ 不再添加 meta_footer

        c = epub.EpubHtml(title=c_title, file_name=f"chap_{ch['id']}.xhtml", lang='zh')
        c.content = f"<h1>{c_title}</h1>{html_content}"

        book.add_item(c)
        epub_chapters.append(c)

    book.toc = (epub_chapters)
    book.add_item(epub.EpubNcx())
    book.add_item(epub.EpubNav())
    book.spine = ['nav'] + epub_chapters

    epub_filename = f"{TOPIC}_full.epub"
    epub.write_epub(epub_filename, book)
    print(f"✅ EPUB 电子书已生成: {epub_filename}")
    return epub_filename

# 用于存储生成的文件名，供后续发送邮件使用
generated_files = {
    'txt_file': None,
    'epub_file': None,
    'success': False
}

if __name__ == "__main__":
    start_time = time.time()
    history_facts = run_step1_historian()
    if history_facts:
        outline = run_step2_architect(history_facts)
        if outline:
            print("\n✅ 分场大纲已生成 (预览前3章):")
            for ch in outline[:3]:
                print(f"   - {ch.get('title')}: {ch.get('mission_objective')}")

            final_script_str, chapters_struct = run_step3_director(outline)

            if final_script_str:
                txt_filename = f"{TOPIC}_final_script.txt"
                with open(txt_filename, "w", encoding="utf-8") as f:
                    f.write(f"【主题】：{TOPIC}\n【总字数】：{len(final_script_str)}\n")
                    f.write(final_script_str)

                print(f"\n🎉🎉🎉 全部完成！耗时: {int(time.time() - start_time)}秒")
                print(f"📄 TXT 文件 (含元数据): {txt_filename}")
                generated_files['txt_file'] = txt_filename

                if chapters_struct:
                    epub_name = run_step4_publisher(chapters_struct)
                    print(f"📘 EPUB 文件 (纯净版): {epub_name}")
                    generated_files['epub_file'] = epub_name

                generated_files['success'] = True
            else:
                print("❌ 正文生成失败")
        else:
            print("❌ 大纲生成失败")
    else:
        print("❌ 史料挖掘失败")


🕵️ [Step 1] 正在挖掘【马绍尔群岛的历史】的高密度历史素材...


In [ ]:
# @title 发送邮件到指定邮箱

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email.header import Header
from email import encoders
import os

def get_mime_type(filename):
    """根据文件扩展名返回 MIME 类型"""
    ext = os.path.splitext(filename)[1].lower()
    mime_types = {
        '.txt': 'text/plain',
        '.epub': 'application/epub+zip',
        '.pdf': 'application/pdf',
        '.zip': 'application/zip',
    }
    return mime_types.get(ext, 'application/octet-stream')

def encode_filename(filename):
    """编码中文文件名（使用 RFC 2231 标准）"""
    # 使用 RFC 2231 编码，这是邮件附件中文文件名的标准方法
    # 方法：将 UTF-8 编码的字节转换为百分号编码
    utf8_bytes = filename.encode('utf-8')
    # 将每个字节转换为 %XX 格式（除了 ASCII 可打印字符）
    encoded_parts = []
    for byte in utf8_bytes:
        if 0x20 <= byte <= 0x7E and byte not in (0x22, 0x5C):  # 可打印 ASCII 字符（除了引号和反斜杠）
            encoded_parts.append(chr(byte))
        else:
            encoded_parts.append(f'%{byte:02X}')
    encoded = ''.join(encoded_parts)
    return f"filename*=UTF-8''{encoded}"

def send_email_with_attachments(to_email, sender_email, sender_password, subject, body, attachments):
    """
    发送带附件的邮件

    参数:
    - to_email: 收件人邮箱
    - sender_email: 发件人邮箱（QQ邮箱）
    - sender_password: 发件人授权码（不是登录密码，需要在QQ邮箱设置中获取）
    - subject: 邮件主题
    - body: 邮件正文
    - attachments: 附件文件路径列表
    """
    try:
        # 创建邮件对象
        msg = MIMEMultipart()
        msg['From'] = sender_email
        msg['To'] = to_email
        # 编码邮件主题（支持中文）
        msg['Subject'] = Header(subject, 'utf-8')

        # 添加邮件正文
        msg.attach(MIMEText(body, 'plain', 'utf-8'))

        # 添加附件
        for file_path in attachments:
            if file_path and os.path.exists(file_path):
                filename = os.path.basename(file_path)
                mime_type = get_mime_type(filename)

                with open(file_path, 'rb') as f:
                    part = MIMEBase('application', 'octet-stream')
                    part.set_payload(f.read())
                    encoders.encode_base64(part)

                    # 正确编码中文文件名（使用 RFC 2231 标准）
                    # 生成 UTF-8 编码的文件名（RFC 2231 格式）
                    encoded_filename = encode_filename(filename)
                    # 同时提供 ASCII 备用文件名（用于兼容旧版邮件客户端）
                    # 如果文件名包含非 ASCII 字符，使用简化版本
                    try:
                        ascii_fallback = filename.encode('ascii').decode('ascii')
                        # 如果全是 ASCII，使用简单格式
                        disposition = f'attachment; filename="{ascii_fallback}"; {encoded_filename}'
                    except UnicodeEncodeError:
                        # 包含非 ASCII 字符，只使用 RFC 2231 格式
                        disposition = f'attachment; {encoded_filename}'

                    part.add_header('Content-Disposition', disposition)
                    part.add_header('Content-Type', mime_type)

                    msg.attach(part)
                    print(f"✅ 已添加附件: {filename}")

        # 连接QQ邮箱SMTP服务器
        print(f"📧 正在连接QQ邮箱服务器...")
        server = smtplib.SMTP('smtp.qq.com', 587)
        server.starttls()  # 启用TLS加密
        server.login(sender_email, sender_password)

        # 发送邮件
        print(f"📤 正在发送邮件到 {to_email}...")
        text = msg.as_string()
        server.sendmail(sender_email, to_email, text)
        server.quit()

        print(f"✅ 邮件发送成功！")
        return True
    except Exception as e:
        print(f"❌ 邮件发送失败: {str(e)}")
        print(f"💡 提示：请确保已开启QQ邮箱的SMTP服务，并使用授权码（不是登录密码）")
        return False

# ==========================================
# 邮件配置
# ==========================================
# 收件人邮箱
RECIPIENT_EMAIL = "1020493635@qq.com"  # @param {type:"string"}

# 发件人邮箱（需要填写你的QQ邮箱）
SENDER_EMAIL = "1020493635@qq.com"  # @param {type:"string"}

# 发件人授权码（需要在QQ邮箱设置中获取，不是登录密码）
# 获取方法：QQ邮箱 -> 设置 -> 账户 -> 开启SMTP服务 -> 生成授权码
SENDER_AUTHORIZATION_CODE = "rqrlcnkiqamqbcbi"  # @param {type:"string"}

# ==========================================
# 检查并发送邮件
# ==========================================

if generated_files.get('success'):
    # 收集所有生成的文件
    attachments = []
    if generated_files.get('txt_file'):
        attachments.append(generated_files['txt_file'])
    if generated_files.get('epub_file'):
        attachments.append(generated_files['epub_file'])

    if attachments:
        if SENDER_EMAIL and SENDER_AUTHORIZATION_CODE:
            subject = f"时空采样者 - {TOPIC} 生成完成"
            body = f"""
你好！

《时空采样者：{TOPIC}》已生成完成。

生成内容：
- 主题：{TOPIC}
- 章节数：{NUM_CHAPTERS}
- 每章字数：{WORDS_PER_CHAPTER}

附件包含：
{chr(10).join([f"- {os.path.basename(f)}" for f in attachments if f])}

请查收附件。

---
此邮件由时空采样者自动发送
"""
            send_email_with_attachments(
                to_email=RECIPIENT_EMAIL,
                sender_email=SENDER_EMAIL,
                sender_password=SENDER_AUTHORIZATION_CODE,
                subject=subject,
                body=body,
                attachments=attachments
            )
        else:
            print("⚠️ 请先填写 SENDER_EMAIL 和 SENDER_AUTHORIZATION_CODE")
            print("💡 获取授权码：QQ邮箱 -> 设置 -> 账户 -> 开启SMTP服务 -> 生成授权码")
    else:
        print("⚠️ 没有找到生成的文件")
else:
    print("⚠️ 程序尚未成功运行，请先运行主程序 cell")